In [0]:
%sql
-- Elimina todos los registros de la tabla silver para cargar datos limpios
TRUNCATE TABLE airline_catalog.silver.flights_silver;

-- Inserta datos limpios y validados desde la tabla bronze a la tabla silver
INSERT INTO airline_catalog.silver.flights_silver
(
    flight_date,                -- Fecha del vuelo
    airline_code,               -- Código de la aerolínea
    tail_number,                -- Matrícula del avión
    flight_number,              -- Número de vuelo
    origin_airport_id,          -- ID del aeropuerto de origen
    origin_code,                -- Código del aeropuerto de origen
    origin_city,                -- Ciudad de origen
    origin_state,               -- Estado de origen
    destination_airport_id,     -- ID del aeropuerto de destino
    destination_code,           -- Código del aeropuerto de destino
    destination_city,           -- Ciudad de destino
    destination_state,          -- Estado de destino
    scheduled_departure_time,   -- Hora programada de salida
    actual_departure_time,      -- Hora real de salida
    departure_delay,            -- Minutos de retraso en la salida
    scheduled_arrival_time,     -- Hora programada de llegada
    actual_arrival_time,        -- Hora real de llegada
    arrival_delay,              -- Minutos de retraso en la llegada
    cancelled,                  -- Indicador de vuelo cancelado
    cancellation_code,          -- Código de cancelación
    diverted,                   -- Indicador de vuelo desviado
    actual_elapsed_time,        -- Tiempo real transcurrido del vuelo
    air_time,                   -- Tiempo en el aire
    distance,                   -- Distancia recorrida
    carrier_delay,              -- Retraso por la aerolínea
    weather_delay,              -- Retraso por condiciones meteorológicas
    nas_delay,                  -- Retraso por el sistema nacional de aviación
    security_delay,             -- Retraso por seguridad
    late_aircraft_delay,        -- Retraso por llegada tardía de la aeronave
    route,                      -- Ruta del vuelo (origen-destino)
    flight_status,              -- Estado del vuelo (Cancelado, Retrasado, A tiempo)
    _source_table,              -- Nombre de la tabla fuente
    _processing_timestamp       -- Timestamp de procesamiento
)

SELECT
    -- Conversión de la fecha del vuelo desde string a tipo fecha
    TO_DATE(FL_DATE, 'M/d/yyyy h:mm:ss a') AS flight_date,

    -- Código de la aerolínea
    OP_UNIQUE_CARRIER AS airline_code,

    -- Matrícula del avión
    TAIL_NUM AS tail_number,

    -- Número de vuelo
    OP_CARRIER_FL_NUM AS flight_number,

    -- ID del aeropuerto de origen
    ORIGIN_AIRPORT_ID,

    -- Código del aeropuerto de origen
    ORIGIN AS origin_code,

    -- Ciudad de origen
    ORIGIN_CITY_NAME AS origin_city,

    -- Estado de origen
    ORIGIN_STATE_NM AS origin_state,

    -- ID del aeropuerto de destino
    DEST_AIRPORT_ID,

    -- Código del aeropuerto de destino
    DEST AS destination_code,

    -- Ciudad de destino
    DEST_CITY_NAME AS destination_city,

    -- Estado de destino
    DEST_STATE_NM AS destination_state,

    -- Hora programada de salida
    CRS_DEP_TIME AS scheduled_departure_time,

    -- Hora real de salida
    DEP_TIME AS actual_departure_time,

    -- Minutos de retraso en la salida
    DEP_DELAY_NEW AS departure_delay,

    -- Hora programada de llegada
    CRS_ARR_TIME AS scheduled_arrival_time,

    -- Hora real de llegada
    ARR_TIME AS actual_arrival_time,

    -- Minutos de retraso en la llegada
    ARR_DELAY_NEW AS arrival_delay,

    -- Indicador booleano de vuelo cancelado
    CASE 
        WHEN CANCELLED = 1 THEN TRUE 
        ELSE FALSE 
    END AS cancelled,

    -- Código de cancelación
    CANCELLATION_CODE,

    -- Indicador booleano de vuelo desviado
    CASE 
        WHEN DIVERTED = 1 THEN TRUE 
        ELSE FALSE 
    END AS diverted,

    -- Tiempo real transcurrido del vuelo
    ACTUAL_ELAPSED_TIME,

    -- Tiempo en el aire
    AIR_TIME,

    -- Distancia recorrida
    DISTANCE,

    -- Retraso por la aerolínea
    CARRIER_DELAY,

    -- Retraso por condiciones meteorológicas
    WEATHER_DELAY,

    -- Retraso por el sistema nacional de aviación
    NAS_DELAY,

    -- Retraso por seguridad
    SECURITY_DELAY,

    -- Retraso por llegada tardía de la aeronave
    LATE_AIRCRAFT_DELAY,

    -- Ruta del vuelo (origen-destino)
    CONCAT(ORIGIN,'-',DEST) AS route,

    -- Estado del vuelo: Cancelado, Retrasado o A tiempo
    CASE
        WHEN CANCELLED = 1 THEN 'Cancelled'
        WHEN ARR_DELAY_NEW > 0 THEN 'Delayed'
        ELSE 'On Time'
    END AS flight_status,

    -- Nombre de la tabla fuente
    'airline_catalog.bronze.flights_bronze' AS _source_table,

    -- Timestamp de procesamiento actual
    CURRENT_TIMESTAMP() AS _processing_timestamp

FROM airline_catalog.bronze.flights_bronze;

In [0]:
select * from airline_catalog.silver.flights_silver limit 10 

In [0]:
SELECT COUNT(*) FROM airline_catalog.silver.flights_silver